# torch.compile Control APIs — Fine-Grained Compilation

**What you'll learn:**
- Compiler stances: control how `torch.compile` handles recompilation
- `@torch.compiler.disable` — skip compilation for specific functions
- `@torch.compiler.allow_in_graph` — treat functions as opaque nodes
- `substitute_in_graph` — replace untraceable functions during compilation
- `mark_dynamic` / `mark_static` — explicit shape control
- Graph breaks: what they are, how to find and fix them
- `torch._dynamo.explain()` — get a compilation diagnostic report
- `TORCH_LOGS` — debug compilation at various verbosity levels

**Prerequisites:** Familiarity with `torch.compile` basics.

**All code runs on CPU** using the "eager" backend for demonstration.

In [ ]:
import torch
import torch.nn as nn
import torch._dynamo
import torch.compiler

print(f"PyTorch version: {torch.__version__}")
torch._dynamo.reset()  # Clean state for demonstrations

## Compiler Stances — Control Recompilation Behavior

A **stance** tells the compiler how to react when it encounters code that would trigger recompilation (e.g., new tensor shapes). Available stances:

| Stance | Behavior |
|--------|----------|
| `"default"` | Normal: compile, recompile as needed |
| `"force_eager"` | Never compile — run everything eagerly |
| `"eager_on_recompile"` | Compile once, run eagerly if shapes change |
| `"fail_on_recompile"` | Compile once, raise error if recompilation needed |

In [ ]:
torch._dynamo.reset()

@torch.compile(backend="eager")
def my_fn(x):
    return x.sin() + x.cos()

# With default stance — compiles normally
with torch.compiler.set_stance("default"):
    result = my_fn(torch.randn(4))
    print(f"default stance: compiled and ran, result shape = {result.shape}")

# force_eager — skips compilation entirely
torch._dynamo.reset()
with torch.compiler.set_stance("force_eager"):
    result = my_fn(torch.randn(4))
    print(f"force_eager stance: ran eagerly (no compilation)")

# eager_on_recompile — compile once, then fall back
torch._dynamo.reset()
with torch.compiler.set_stance("eager_on_recompile"):
    result1 = my_fn(torch.randn(4))    # compiles for shape [4]
    result2 = my_fn(torch.randn(8))    # different shape → runs eagerly instead of recompiling
    print(f"eager_on_recompile: shape [4] compiled, shape [8] ran eagerly")

## @torch.compiler.disable — Skip Compilation for Specific Functions

Some functions contain code that Dynamo can't trace (custom C extensions, complex control flow, debugging code). Decorate them with `@torch.compiler.disable` to make the compiler skip them.

In [ ]:
torch._dynamo.reset()

@torch.compiler.disable
def my_custom_logging(x, step):
    """This function uses Python features that can't be compiled."""
    import logging
    logging.debug(f"Step {step}: tensor norm = {x.norm().item()}")
    return x

@torch.compile(backend="eager")
def train_step(x, step):
    x = x * 2 + 1
    x = my_custom_logging(x, step)  # graph break here, but that's OK
    return x.relu()

result = train_step(torch.randn(4), step=1)
print(f"@torch.compiler.disable lets untraceable code coexist with compiled code")
print(f"Result: {result}")

## @torch.compiler.allow_in_graph — Opaque Graph Nodes

`allow_in_graph` tells the compiler: "trust me, this function is safe to include in the graph without tracing into it." The function becomes an opaque node — the compiler won't look inside but won't break the graph either.

In [ ]:
torch._dynamo.reset()

@torch.compiler.allow_in_graph
def my_special_activation(x):
    """A custom activation — compiler includes it as-is in the graph."""
    return torch.where(x > 0, x, 0.1 * (torch.exp(x) - 1))

# Track what the compiler captures
graphs = []
def capture_backend(gm, example_inputs):
    graphs.append(gm)
    return gm.forward

@torch.compile(backend=capture_backend)
def model_fn(x):
    x = x @ torch.randn(8, 8)
    x = my_special_activation(x)  # opaque node — no graph break!
    return x.sum()

result = model_fn(torch.randn(4, 8))
print(f"Number of graph segments: {len(graphs)} (1 = no graph breaks!)")
print(f"\nGraph contains my_special_activation as opaque node:")
print(graphs[0].code)

## substitute_in_graph — Replace Untraceable Functions

`substitute_in_graph` lets you provide a traceable replacement for a function that the compiler can't trace. The original runs at eager time; the substitute runs during compilation.

In [ ]:
torch._dynamo.reset()

# Imagine this uses a C extension internally
def fast_gelu(x):
    """Original function — maybe untraceable."""
    return x * torch.sigmoid(1.702 * x)

# Provide a traceable equivalent for the compiler
def _gelu_substitute(x):
    return x * torch.sigmoid(1.702 * x)

torch.compiler.substitute_in_graph(fast_gelu, _gelu_substitute)

graphs = []
def capture_backend(gm, example_inputs):
    graphs.append(gm)
    return gm.forward

@torch.compile(backend=capture_backend)
def use_gelu(x):
    return fast_gelu(x * 2)

result = use_gelu(torch.randn(4))
print(f"substitute_in_graph: compiler used the substitute, no graph break")
print(f"Graph segments: {len(graphs)}")
print(f"Result: {result}")

## mark_dynamic / mark_static — Shape Control

By default, `torch.compile` specializes on tensor shapes. If shapes change, it recompiles. You can control this:
- `torch._dynamo.mark_dynamic(tensor, dim)` — tell compiler this dimension varies
- `torch._dynamo.mark_static(tensor, dim)` — tell compiler this dimension is fixed

In [ ]:
torch._dynamo.reset()

compile_count = 0

def counting_backend(gm, example_inputs):
    global compile_count
    compile_count += 1
    return gm.forward

# Problem: recompilation on every new shape
@torch.compile(backend=counting_backend)
def process(x):
    return x.sum(dim=-1)

compile_count = 0
for seq_len in [10, 20, 30, 40]:
    x = torch.randn(2, seq_len, 64)
    process(x)
print(f"Without mark_dynamic: compiled {compile_count} times (once per shape!)")

# Solution: mark the sequence dimension as dynamic
torch._dynamo.reset()
compile_count = 0

@torch.compile(backend=counting_backend)
def process_dynamic(x):
    return x.sum(dim=-1)

for seq_len in [10, 20, 30, 40]:
    x = torch.randn(2, seq_len, 64)
    torch._dynamo.mark_dynamic(x, 1)  # dim 1 (seq_len) is dynamic
    process_dynamic(x)
print(f"With mark_dynamic:    compiled {compile_count} time(s) (handles all shapes!)")

## Graph Breaks and fullgraph=True

A **graph break** splits compiled code into multiple graph segments with Python execution between them. This hurts performance. Use `fullgraph=True` to catch them early.

In [ ]:
torch._dynamo.reset()

# This function has a graph break (print causes it)
def fn_with_break(x):
    x = x * 2
    print("debug")  # graph break!
    return x + 1

# Without fullgraph — silently creates multiple segments
compiled_fn = torch.compile(fn_with_break, backend="eager")
result = compiled_fn(torch.tensor([1.0, 2.0]))
print(f"Result (with graph break): {result}")

# With fullgraph=True — raises an error on graph breaks
try:
    compiled_strict = torch.compile(fn_with_break, backend="eager", fullgraph=True)
    compiled_strict(torch.tensor([1.0, 2.0]))
except Exception as e:
    print(f"\nfullgraph=True caught the break:")
    print(f"  {type(e).__name__}: {str(e)[:100]}...")

## torch.compiler.assume_constant_result

Tells the compiler that a function always returns the same result, so it can constant-fold the call. Useful for configuration lookups or environment checks.

In [ ]:
torch._dynamo.reset()

@torch.compiler.assume_constant_result
def get_config():
    """Pretend this reads from a config file — result never changes at runtime."""
    return {"scale": 2.0, "bias": 0.5}

@torch.compile(backend="eager", fullgraph=True)
def configured_fn(x):
    cfg = get_config()  # compiler treats this as a constant — no graph break
    return x * cfg["scale"] + cfg["bias"]

result = configured_fn(torch.tensor([1.0, 2.0, 3.0]))
print(f"assume_constant_result: config lookup compiled without graph break")
print(f"Result: {result}")
print(f"Expected: {torch.tensor([1.0, 2.0, 3.0]) * 2.0 + 0.5}")

## torch._dynamo.explain() — Compilation Diagnostic Report

`explain()` runs your function and reports what the compiler did: how many graph breaks, why, and which ops are in each segment.

In [ ]:
torch._dynamo.reset()

def complex_fn(x):
    x = x.sin()
    x = x + torch.randn_like(x)
    print("logging")  # causes graph break
    x = x.cos()
    return x

explanation = torch._dynamo.explain(complex_fn)(torch.randn(4, 4))
print(explanation)

## CompileCounter — Test Backend for Counting Compilations

The `CompileCounterAndRecorder` from PyTorch internals (or a simple custom version) is useful for testing how many times the compiler fires.

In [ ]:
torch._dynamo.reset()

class CompileCounter:
    """Minimal compile-counting backend for testing."""
    def __init__(self):
        self.frame_count = 0
        self.op_count = 0
        self.graphs = []

    def __call__(self, gm, example_inputs):
        self.frame_count += 1
        self.op_count += len([n for n in gm.graph.nodes if n.op == "call_function"])
        self.graphs.append(gm)
        return gm.forward

counter = CompileCounter()

@torch.compile(backend=counter)
def fn(x, y):
    a = x.sin()
    b = y.cos()
    return a + b

# First call — compiles
fn(torch.randn(4), torch.randn(4))
print(f"After 1st call: {counter.frame_count} compilation(s), {counter.op_count} ops")

# Same shapes — no recompilation
fn(torch.randn(4), torch.randn(4))
print(f"After 2nd call: {counter.frame_count} compilation(s) (cached!)")

# Different shapes — recompiles
fn(torch.randn(8), torch.randn(8))
print(f"After 3rd call (new shape): {counter.frame_count} compilation(s)")

## EagerAndRecordGraphs — Inspect the FX Graphs

For debugging, you can use a backend that records the FX graphs the compiler produces. This shows exactly what ops end up in each compiled region.

In [ ]:
torch._dynamo.reset()

recorded_graphs = []

def record_backend(gm, example_inputs):
    """Backend that records graphs and runs eagerly."""
    recorded_graphs.append(gm)
    return gm.forward

@torch.compile(backend=record_backend)
def mlp(x):
    x = torch.nn.functional.linear(x, torch.randn(16, 8))
    x = torch.relu(x)
    x = torch.nn.functional.linear(x, torch.randn(4, 16))
    return x

recorded_graphs.clear()
result = mlp(torch.randn(2, 8))

print(f"Captured {len(recorded_graphs)} graph(s)")
print(f"\nFX Graph code:")
print(recorded_graphs[0].code)

## TORCH_LOGS — Debug Compilation from the Environment

PyTorch provides extensive logging for debugging compilation. Key log settings:

| Setting | What it shows |
|---------|--------------|
| `TORCH_LOGS="+dynamo"` | Dynamo tracing decisions |
| `TORCH_LOGS="graph_breaks"` | Only graph break reasons |
| `TORCH_LOGS="recompiles"` | Why recompilation happened |
| `TORCH_LOGS="guards"` | Guard conditions for cached code |
| `TORCH_LOGS="+inductor"` | Inductor code generation |

You can also set these programmatically:

In [ ]:
import logging
torch._dynamo.reset()

# Programmatic logging (equivalent to TORCH_LOGS="recompiles")
torch._logging.set_logs(recompiles=True)

@torch.compile(backend="eager")
def recompile_demo(x):
    return x.relu()

# First call — compiles
recompile_demo(torch.randn(4))
# Second call with different shape — triggers recompile log
recompile_demo(torch.randn(8))

# Reset logging
torch._logging.set_logs()
print("\n(Above: recompile logging shows WHY recompilation happened)")

## Putting It Together: A Real Model with Compilation Controls

In [ ]:
torch._dynamo.reset()

@torch.compiler.disable
def log_stats(name, tensor):
    """Debugging helper — not compiled."""
    print(f"  [{name}] mean={tensor.mean().item():.3f}, std={tensor.std().item():.3f}")

class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear1 = nn.Linear(32, 64)
        self.linear2 = nn.Linear(64, 10)

    def forward(self, x):
        x = self.linear1(x).relu()
        log_stats("after_relu", x)
        x = self.linear2(x)
        return x

model = torch.compile(MyModel(), backend="eager")
x = torch.randn(4, 32)
output = model(x)
print(f"\nModel output shape: {output.shape}")
print("The disabled function runs normally while the rest is compiled")

## Try It Yourself

**Exercise:** Find and fix the graph break in this function.

The goal: make it compile with `fullgraph=True` (no graph breaks).

Hints:
- Use `torch._dynamo.explain()` to identify the break
- Replace the problematic code with a compiler-friendly alternative

In [ ]:
torch._dynamo.reset()

# This function has a graph break — find and fix it!
def broken_fn(x):
    x = x * 2
    # This line causes a graph break:
    x_list = x.tolist()  # converts to Python list — graph break!
    x = x + 1
    return x.relu()

# Step 1: Use explain to find the break
explanation = torch._dynamo.explain(broken_fn)(torch.randn(4))
print("DIAGNOSIS:")
print(explanation)

# Step 2: Fix it! (hint: remove or replace the tolist() call)
# YOUR FIX HERE:
def fixed_fn(x):
    x = x * 2
    # removed the .tolist() — it wasn't needed for the computation
    x = x + 1
    return x.relu()

# Verify fix works with fullgraph=True
torch._dynamo.reset()
compiled_fixed = torch.compile(fixed_fn, backend="eager", fullgraph=True)
result = compiled_fixed(torch.randn(4))
print(f"\nFixed! Result: {result}")

## Key Takeaways

1. **Compiler stances** (`set_stance`) control recompilation behavior — use `"eager_on_recompile"` in production to avoid surprise latency spikes
2. **`@torch.compiler.disable`** — marks functions to skip during compilation (causes a graph break, but intentionally)
3. **`@torch.compiler.allow_in_graph`** — includes a function as an opaque node without tracing into it (no graph break)
4. **`substitute_in_graph`** — provides a traceable stand-in for an untraceable function
5. **`mark_dynamic(tensor, dim)`** — prevents recompilation when a dimension varies (essential for variable-length sequences)
6. **`fullgraph=True`** — fails loudly on graph breaks; use during development to ensure clean compilation
7. **`assume_constant_result`** — tells the compiler a function's return value never changes
8. **`torch._dynamo.explain()`** — diagnostic tool showing graph breaks, their causes, and compiled regions
9. **CompileCounter** — lightweight test backend for verifying compilation count
10. **TORCH_LOGS** — environment variable for detailed compilation debugging (`graph_breaks`, `recompiles`, `guards`)